# ABLE Distillation: Local Fine-Tuning

Universal notebook for fine-tuning ABLE student models. Works in **VS Code Jupyter**, **JupyterLab**, or any local notebook environment.

**Auto-detects hardware:**
- **CUDA** (NVIDIA GPU): Uses Unsloth for 2x faster training
- **MPS** (Apple Silicon): Uses PEFT + float16 (Unsloth requires CUDA)
- **CPU**: float32 fallback (testing only, very slow)

**Free Colab T4 models** (12-24h runtime, the ones we prioritize):

| Model | Base | Params | GGUF Size | VRAM |
|-------|------|--------|-----------|------|
| **`able-gemma4-e4b`** | Gemma 4 E4B | 5.1B (4.5B effective) | ~3GB q4_k_m | **8-10GB** |
| `able-nano-9b` | Qwen 3.5 9B | 9B | ~5GB q4_k_m | 12GB |

**Server models** (need A100/H100):

| Model | Base | VRAM |
|-------|------|------|
| `able-gemma4-31b` | Gemma 4 31B | 22GB+ |
| `able-student-27b` | Qwen 3.5 27B | 24GB+ |

**Recommended flow:** Train `able-gemma4-e4b` on free Colab T4 → download GGUF → `ollama create` → done.

For **Apple Silicon MLX** training, see `notebooks/train_mlx_*.sh`.
For **Colab** (remote GPU), see `notebooks/unsloth_finetune_*.ipynb`.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell before running
# ═══════════════════════════════════════════════════════════

# Model selection — E4B is the primary model (smallest, fastest, free T4)
MODEL_NAME = "able-gemma4-e4b"      # Gemma 4 E4B edge — 8GB VRAM, Apache 2.0
# MODEL_NAME = "able-nano-9b"       # Qwen 3.5 9B edge — 12GB VRAM
# MODEL_NAME = "able-student-27b"   # Qwen 3.5 27B server — 24GB+ VRAM
# MODEL_NAME = "able-gemma4-31b"    # Gemma 4 31B server — 22GB+ VRAM

# Corpus path — uses latest harvest output
# Local repo: data/corpus/default/v048/train.jsonl
# Harvester output: ~/.able/distillation/corpus/default/v{latest}/train.jsonl
import os, glob
_corpus_candidates = sorted(glob.glob(os.path.expanduser(
    "~/.able/distillation/corpus/default/v*/train.jsonl")))
CORPUS_PATH = _corpus_candidates[-1] if _corpus_candidates else "data/corpus/default/v048/train.jsonl"

# Training hyperparameters (auto-adjusted per device in next cell)
EPOCHS = 3
LEARNING_RATE = 2e-4

# GGUF quantization methods for export
QUANT_METHODS = ["q4_k_m", "iq2_m"]  # E4B: compact + ultra-compact

# HuggingFace Hub (optional push)
HF_REPO = None  # e.g., "able-distilled/able-gemma4-e4b"
HF_TOKEN = None  # e.g., "hf_..."

print(f"Model: {MODEL_NAME}")
print(f"Corpus: {CORPUS_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# ENVIRONMENT DETECTION — Auto-detects device + memory
# ═══════════════════════════════════════════════════════════
import sys
import platform
import subprocess
from pathlib import Path

# Model configs (inline so notebook is standalone)
MODEL_CONFIGS = {
    "able-nano-9b": {
        "base_model": "Qwen/Qwen3.5-9B",
        "lora_r": 16, "lora_alpha": 16,
        "max_seq_length": 2048,
        "is_gemma4": False,
    },
    "able-student-27b": {
        "base_model": "Qwen/Qwen3.5-27B",
        "lora_r": 32, "lora_alpha": 32,
        "max_seq_length": 4096,
        "is_gemma4": False,
    },
    "able-gemma4-31b": {
        "base_model": "google/gemma-4-31b-it",
        "lora_r": 8, "lora_alpha": 8,
        "max_seq_length": 8192,
        "is_gemma4": True,
    },
    "able-gemma4-e4b": {
        "base_model": "google/gemma-4-E4B-it",
        "lora_r": 8, "lora_alpha": 8,
        "max_seq_length": 4096,
        "is_gemma4": True,
    },
}

cfg = MODEL_CONFIGS[MODEL_NAME]
BASE_MODEL = cfg["base_model"]
MAX_SEQ_LENGTH = cfg["max_seq_length"]

print("=" * 60)
print("  ABLE Distillation — Environment Detection")
print("=" * 60)

DEVICE = "cpu"
DEVICE_NAME = platform.processor() or "unknown"
MEMORY_GB = 0.0
USE_UNSLOTH = False
DTYPE = "float32"
CHIP_NAME = ""

try:
    import torch
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        DEVICE = "mps"
        try:
            r = subprocess.run(["sysctl", "-n", "hw.memsize"],
                               capture_output=True, text=True, timeout=5)
            MEMORY_GB = round(int(r.stdout.strip()) / (1024**3), 1)
        except Exception:
            MEMORY_GB = 16.0
        try:
            r = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                               capture_output=True, text=True, timeout=5)
            CHIP_NAME = r.stdout.strip()
        except Exception:
            CHIP_NAME = "Apple Silicon"
        DEVICE_NAME = CHIP_NAME
        DTYPE = "float16"  # MPS float16 works, bfloat16 does not
        print(f"  Device:  MPS (Apple Silicon)")
        print(f"  Chip:    {CHIP_NAME}")
        print(f"  Memory:  {MEMORY_GB} GB unified")
        print(f"  Dtype:   float16 (bfloat16 unreliable on MPS)")
        print(f"  Backend: PEFT LoRA (Unsloth requires CUDA)")
    elif torch.cuda.is_available():
        DEVICE = "cuda"
        DEVICE_NAME = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        MEMORY_GB = round(props.total_mem / (1024**3), 1)
        cap = torch.cuda.get_device_capability(0)
        DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
        USE_UNSLOTH = True
        print(f"  Device:  CUDA")
        print(f"  GPU:     {DEVICE_NAME}")
        print(f"  VRAM:    {MEMORY_GB} GB")
        print(f"  Dtype:   {DTYPE}")
        print(f"  Backend: Unsloth (2x faster training)")
    else:
        DTYPE = "float32"
        print(f"  Device:  CPU (slow — use for testing only)")
        print(f"  Proc:    {DEVICE_NAME}")
except ImportError:
    print("  WARNING: PyTorch not installed. Will install in next cell.")

# Auto-adjust batch size by memory
if MEMORY_GB >= 64:
    BATCH_SIZE, GRAD_ACCUM = 8, 2
elif MEMORY_GB >= 32:
    BATCH_SIZE, GRAD_ACCUM = 4, 4
elif MEMORY_GB >= 16:
    BATCH_SIZE, GRAD_ACCUM = 2, 8
else:
    BATCH_SIZE, GRAD_ACCUM = 1, 16

# MPS needs more conservative settings for large models
if DEVICE == "mps" and "27b" in MODEL_NAME or "31b" in MODEL_NAME:
    BATCH_SIZE = max(1, BATCH_SIZE // 2)
    GRAD_ACCUM = min(32, GRAD_ACCUM * 2)
    if MEMORY_GB < 48:
        print(f"  WARNING: {MODEL_NAME} may not fit in {MEMORY_GB}GB. "
              f"Consider able-nano-9b or able-gemma4-e4b instead.")

print(f"  Batch:   {BATCH_SIZE} x {GRAD_ACCUM} grad accum")
print(f"  Unsloth: {'Yes' if USE_UNSLOTH else 'No'}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════
import subprocess
import sys

def pip_install(*packages):
    """Install packages using the current Python interpreter."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + list(packages)
    )

if USE_UNSLOTH:
    print("Installing Unsloth + dependencies (CUDA path)...")
    # Pin >=2026.4.3 for Gemma 4 gradient accumulation fix
    pip_install("unsloth>=2026.4.3")
    pip_install("--no-deps", "trl", "peft", "accelerate", "bitsandbytes")
else:
    print("Installing PEFT + dependencies (MPS/CPU path)...")
    pip_install("torch", "transformers", "peft", "trl", "accelerate", "datasets")
    # bitsandbytes MPS support is experimental — skip on MPS
    if DEVICE == "cuda":
        pip_install("bitsandbytes")

# Reduce HuggingFace API calls (Unsloth Studio finding: 90% fewer throttles)
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print("Dependencies installed.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════════════
import torch

if USE_UNSLOTH:
    # ── CUDA path: Unsloth (2x faster, 70% less VRAM) ────────
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,  # Auto-detect (float16 on T4, bfloat16 on A100/H100)
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg["lora_r"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=0,
        use_gradient_checkpointing="unsloth",
    )

else:
    # ── MPS/CPU path: standard PEFT LoRA ─────────────────────
    # SAFETY: Gemma 4 KV-sharing corrupts with vanilla PEFT + gradient checkpointing
    if cfg["is_gemma4"]:
        print("\n" + "=" * 60)
        print("ERROR: Gemma 4 requires Unsloth (CUDA) for safe training.")
        print("KV-sharing corruption occurs with vanilla PEFT + gradient checkpointing.")
        print("Options:")
        print("  1. Use the Colab notebook (free T4 GPU, 24h runtime)")
        print("       → notebooks/unsloth_finetune_able-gemma4-e4b.ipynb")
        print("  2. Use the MLX script (Apple Silicon native, no PEFT)")
        print("       → notebooks/train_mlx_able-gemma4-e4b.sh")
        print("  3. Use a CUDA machine with Unsloth installed")
        print("=" * 60 + "\n")
        raise RuntimeError("Gemma 4 requires Unsloth (CUDA). See options above.")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # MPS: float16 (not bfloat16), eager attention, no 4-bit
    # CPU: float32
    load_dtype = torch.float16 if DEVICE == "mps" else torch.float32
    load_kwargs = {
        "torch_dtype": load_dtype,
        "attn_implementation": "eager",  # No flash attention on MPS
        "low_cpu_mem_usage": True,
    }

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)

    if DEVICE == "mps":
        model = model.to("mps")

    # Enable gradient checkpointing for memory savings
    model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model: {BASE_MODEL}")
print(f"Trainable: {trainable:,} / {total:,} params ({trainable/total*100:.2f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD & FORMAT CORPUS
# ═══════════════════════════════════════════════════════════
from datasets import load_dataset
from pathlib import Path

# Resolve corpus path
corpus = Path(CORPUS_PATH).expanduser()
if not corpus.exists():
    corpus = Path("../") / CORPUS_PATH
if not corpus.exists():
    raise FileNotFoundError(
        f"Corpus not found at {CORPUS_PATH}. "
        f"Run the harvest pipeline first:\n"
        f"  python -c \"from able.core.distillation.harvest_runner import run_harvest; run_harvest()\""
    )

print(f"Corpus: {corpus.resolve()}")

def format_chatml(example):
    """Format a single training example as ChatML.
    
    Handles both 'messages' (standard) and 'conversations' (ABLE corpus) keys.
    """
    messages = example.get('messages', example.get('conversations', []))
    if not messages:
        return {"text": ""}
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": formatted}

dataset = load_dataset("json", data_files=str(corpus), split="train")
dataset = dataset.map(format_chatml)

# Filter out empty examples (safety)
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
print(f"Loaded {len(dataset)} training examples")

# Preview first example
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}...")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TRAIN
# ═══════════════════════════════════════════════════════════
from trl import SFTTrainer
from transformers import TrainingArguments

# Gemma 4 models have high training loss (10-15) — this is normal
if cfg["is_gemma4"]:
    print("NOTE: Gemma 4 training loss in range 10-15 is NORMAL.")
    print("Do NOT stop training early thinking loss is diverging.")

# Device-specific TrainingArguments
training_kwargs = {
    "per_device_train_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "num_train_epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": 100,
    "output_dir": f"outputs/{MODEL_NAME}",
    "optim": "adamw_torch",  # adamw_8bit needs bitsandbytes (CUDA only)
    "warmup_steps": 5,
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 42,
    "report_to": "none",  # Disable wandb/tensorboard unless configured
}

if DEVICE == "cuda":
    if USE_UNSLOTH:
        from unsloth import is_bfloat16_supported
        training_kwargs["fp16"] = not is_bfloat16_supported()
        training_kwargs["bf16"] = is_bfloat16_supported()
        training_kwargs["optim"] = "adamw_8bit"
    else:
        training_kwargs["fp16"] = DTYPE == "float16"
        training_kwargs["bf16"] = DTYPE == "bfloat16"
elif DEVICE == "mps":
    # MPS: no fp16/bf16 mixed precision — train in float16 model dtype
    training_kwargs["fp16"] = False
    training_kwargs["bf16"] = False
    training_kwargs["dataloader_pin_memory"] = False
    # MPS doesn't support 8-bit optimizers
    training_kwargs["optim"] = "adamw_torch"
else:
    training_kwargs["fp16"] = False
    training_kwargs["bf16"] = False
    training_kwargs["no_cuda"] = True

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(**training_kwargs),
)

print(f"Training {MODEL_NAME} on {DEVICE}...")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Grad Accum: {GRAD_ACCUM}")
print(f"  LR: {LEARNING_RATE}, Seq Len: {MAX_SEQ_LENGTH}")
print()

trainer_stats = trainer.train()
print(f"\nTraining complete. Loss: {trainer_stats.training_loss:.4f}")

## Export to GGUF for Ollama

Exports the fine-tuned model to GGUF format.
- **CUDA + Unsloth**: Direct GGUF export via Unsloth (Dynamic 2.0 quantization)
- **MPS/CPU**: Save merged model, convert via llama.cpp

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPORT MODEL
# ═══════════════════════════════════════════════════════════
import os
os.makedirs(f"outputs/{MODEL_NAME}-gguf", exist_ok=True)

if USE_UNSLOTH:
    # ── CUDA: Unsloth GGUF export (fastest) ──────────────────
    for quant in QUANT_METHODS:
        print(f"Exporting {quant}...")
        model.save_pretrained_gguf(
            f"outputs/{MODEL_NAME}-gguf",
            tokenizer,
            quantization_method=quant,
        )
        print(f"  Done: outputs/{MODEL_NAME}-gguf")

    # Optional: merged GGUF (full model, not just adapter)
    # model.save_pretrained_merged(
    #     f"outputs/{MODEL_NAME}-merged",
    #     tokenizer,
    #     save_method="merged_16bit",
    # )

else:
    # ── MPS/CPU: Merge LoRA + save for llama.cpp conversion ──
    print("Merging LoRA adapter into base model...")
    merged_dir = f"outputs/{MODEL_NAME}-merged"

    # Merge and save
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f"Merged model saved: {merged_dir}")

    # Also save just the LoRA adapter (much smaller)
    adapter_dir = f"outputs/{MODEL_NAME}-adapter"
    # model.save_pretrained(adapter_dir)  # Uncomment to save adapter separately

    print()
    print("To convert to GGUF, install llama.cpp and run:")
    print(f"  git clone https://github.com/ggml-org/llama.cpp")
    print(f"  pip install -r llama.cpp/requirements.txt")
    print(f"  python3 llama.cpp/convert_hf_to_gguf.py {merged_dir} \\")
    print(f"    --outfile outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-f16.gguf \\")
    print(f"    --outtype f16")
    print()
    print("  Then quantize:")
    for quant in QUANT_METHODS:
        print(f"    llama.cpp/llama-quantize outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-f16.gguf "
              f"outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-{quant}.gguf {quant}")

    # Auto-convert if llama.cpp is available
    llama_cpp = Path("llama.cpp/convert_hf_to_gguf.py")
    if llama_cpp.exists():
        import subprocess
        print("\nllama.cpp found — auto-converting...")
        subprocess.run([
            sys.executable, str(llama_cpp), merged_dir,
            "--outfile", f"outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-f16.gguf",
            "--outtype", "f16",
        ], check=True)
        print(f"GGUF exported: outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-f16.gguf")

In [ ]:
# ═══════════════════════════════════════════════════════════
# OLLAMA DEPLOYMENT
# ═══════════════════════════════════════════════════════════

# Detect GGUF filename
if USE_UNSLOTH:
    gguf_file = f"./outputs/{MODEL_NAME}-gguf/unsloth.Q4_K_M.gguf"
else:
    gguf_file = f"./outputs/{MODEL_NAME}-gguf/{MODEL_NAME}-q4_k_m.gguf"

# Template varies by model family
if cfg["is_gemma4"]:
    chat_template = """TEMPLATE \"\"\"{{- if .System }}<start_of_turn>user
{{ .System }}<end_of_turn>
{{ end }}{{- range .Messages }}<start_of_turn>{{ if eq .Role "assistant" }}model{{ else }}{{ .Role }}{{ end }}
{{ .Content }}<end_of_turn>
{{ end }}<start_of_turn>model
\"\"\"
PARAMETER temperature 0.7
PARAMETER flash_attention on
PARAMETER cache_type_k q4_0
PARAMETER cache_type_v q4_0
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<start_of_turn>\""""
else:
    chat_template = """TEMPLATE \"\"\"{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{- end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
{{ .Response }}<|im_end|>\"\"\"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>\""""

modelfile = f"""FROM {gguf_file}
{chat_template}
SYSTEM You are ABLE, an autonomous AI agent.
"""

with open("Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile generated.")
print()
print("Deploy to Ollama:")
print(f"  ollama create {MODEL_NAME} -f Modelfile")
print(f"  ollama run {MODEL_NAME}")
print()
print("Test:")
print(f'  curl http://localhost:11434/api/generate -d \'{{"model":"{MODEL_NAME}","prompt":"Hello ABLE"}}\'') 

## Training Stats for Federation

Records training metrics for the ABLE federation sync.
Quality improvements are shared across the network.

In [ ]:
# ═══════════════════════════════════════════════════════════
# TRAINING STATS
# ═══════════════════════════════════════════════════════════
import json
from datetime import datetime, timezone

stats = {
    "model": MODEL_NAME,
    "base": BASE_MODEL,
    "corpus_path": str(corpus),
    "corpus_size": len(dataset),
    "epochs": EPOCHS,
    "device": DEVICE,
    "device_name": DEVICE_NAME,
    "memory_gb": MEMORY_GB,
    "backend": "unsloth" if USE_UNSLOTH else "peft",
    "dtype": DTYPE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "quants": QUANT_METHODS,
    "loss": trainer_stats.training_loss,
    "runtime_seconds": trainer_stats.metrics.get("train_runtime", 0),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

os.makedirs(f"outputs", exist_ok=True)
with open(f"outputs/{MODEL_NAME}_training_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))

In [ ]:
# ═══════════════════════════════════════════════════════════
# OPTIONAL: Push to HuggingFace Hub
# ═══════════════════════════════════════════════════════════
if HF_REPO and HF_TOKEN:
    if USE_UNSLOTH:
        model.push_to_hub_gguf(
            HF_REPO,
            tokenizer,
            quantization_method=QUANT_METHODS[0],
            token=HF_TOKEN,
        )
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=f"outputs/{MODEL_NAME}-merged",
            repo_id=HF_REPO,
            repo_type="model",
        )
    print(f"Pushed to {HF_REPO}")
else:
    print("Set HF_REPO and HF_TOKEN in the config cell to push to HuggingFace Hub.")